In [1]:
%pip install python-dotenv pandas neo4j langchain-neo4j langchain-google-genai langchain-experimental langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import pandas as pd
from urllib.parse import urlparse
from dotenv import load_dotenv
from neo4j import GraphDatabase
from langchain_groq import ChatGroq
from langchain_neo4j import Neo4jGraph

# --- FASE 1: LOAD KREDENSIAL & TES KONEKSI ---
print("Memuat kredensial dari .env...")
load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

try:
    # Coba ketuk pintu database Neo4j Sandbox
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print("✅ Koneksi ke Neo4j Sandbox Berhasil!")
    driver.close()
    
    # Inisialisasi LangChain Graph & LLM
    graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)
    llm = ChatGroq(
    temperature=0, 
    model_name="llama-3.3-70b-versatile", 
    api_key=os.getenv("GROQ_API_KEY")
)
    print("✅ LangChain & Groq siap digunakan!")
    
except Exception as e:
    print("❌ Koneksi Gagal! Cek kembali isi file .env milikmu.")
    print(f"Error detail: {e}")

print("-" * 40)

Memuat kredensial dari .env...
✅ Koneksi ke Neo4j Sandbox Berhasil!
✅ LangChain & Groq siap digunakan!
----------------------------------------


In [ ]:
# --- FASE 2: LOAD DATASET & EKSTRAKSI FITUR ---
print("Memproses dataset malicious_phish.csv...")

try:
    # Load dataset
    df = pd.read_csv('malicious_phish.csv')
    print(f"✅ Dataset termuat. Total baris asli: {len(df)}")
    
    # Ambil sampel acak sebanyak 30.000 data
    df = pd.read_csv('malicious_phish.csv').sample(n=30000, random_state=42).reset_index(drop=True)
    
    # Fungsi ekstraksi
    def get_domain(url):
        if not url.startswith(('http://', 'https://')):
            url = 'http://' + url
        return urlparse(url).netloc.split(':')[0]

    def get_tld(domain):
        parts = domain.split('.')
        return '.' + parts[-1] if len(parts) > 1 else 'unknown'
        
    # Terapkan ekstraksi
    df_sample['domain'] = df_sample['url'].apply(get_domain)
    df_sample['tld'] = df_sample['domain'].apply(get_tld)
    
    if 'type' in df_sample.columns:
        df_sample.rename(columns={'type': 'label'}, inplace=True)
        
    print("✅ Ekstraksi fitur berhasil. Berikut 5 baris pertama hasilnya:")
    display(df_sample[['url', 'domain', 'tld', 'label']].head())
    
except FileNotFoundError:
    print("❌ File malicious_phish.csv tidak ditemukan di folder ini.")

Memproses dataset malicious_phish.csv...
✅ Dataset termuat. Total baris asli: 651191
✅ Ekstraksi fitur berhasil. Berikut 5 baris pertama hasilnya:


,url,domain,tld,label
536448,http://37.49.226.178/deusbins/deus.sh4,37.49.226.178,.178,malware
40630,medical-dictionary.thefreedictionary.com/Galt+...,medical-dictionary.thefreedictionary.com,.com,benign
630496,www.jscape.com/sshfactory/,www.jscape.com,.com,phishing
426724,http://www.wsnc.org.au/component/jcalpro/view/983,www.wsnc.org.au,.au,defacement
184034,virtualtourist.com/travel/North_America/Canada...,virtualtourist.com,.com,benign


In [4]:
# --- FASE 3: INGESTION KE NEO4J ---
print("Membangun Knowledge Graph di Neo4j...")

# 1. Buka jalur koneksi (driver)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# 2. Buat Constraint (Aturan agar tidak ada duplikasi data di database)
def create_constraints():
    constraints_queries = [
        "CREATE CONSTRAINT IF NOT EXISTS FOR (u:URL) REQUIRE u.id IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (d:Domain) REQUIRE d.id IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (t:TLD) REQUIRE t.id IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (l:Label) REQUIRE l.id IS UNIQUE"
    ]
    with driver.session() as session:
        for q in constraints_queries:
            session.run(q)
    print("✅ Constraints berhasil diterapkan.")

# 3. Query Cypher Utama untuk Ingestion
# UNWIND berfungsi seperti loop (for each row)
# MERGE berfungsi mencari node/relasi, jika belum ada maka akan dibuat baru
ingest_query = """
UNWIND $data AS row
MERGE (u:URL {id: row.url})
MERGE (d:Domain {id: row.domain})
MERGE (t:TLD {id: row.tld})
MERGE (l:Label {id: row.label})

MERGE (u)-[:HAS_DOMAIN]->(d)
MERGE (u)-[:HAS_TLD]->(t)
MERGE (u)-[:IS_CLASSIFIED_AS]->(l)
"""

# 4. Eksekusi Ingestion
def ingest_data(dataframe):
    # Ubah dataframe menjadi format list of dictionary yang dipahami Neo4j
    records = dataframe.to_dict('records')
    with driver.session() as session:
        session.run(ingest_query, data=records)
    print(f"✅ Berhasil memasukkan {len(records)} baris data ke dalam graf!")

# Jalankan fungsinya
create_constraints()
ingest_data(df_sample)

# Refresh skema graf di LangChain agar LLM tahu struktur terbarunya
graph.refresh_schema()
print("✅ Skema LangChain berhasil diperbarui. Graf siap digunakan!")

Membangun Knowledge Graph di Neo4j...
✅ Constraints berhasil diterapkan.
✅ Berhasil memasukkan 5000 baris data ke dalam graf!
✅ Skema LangChain berhasil diperbarui. Graf siap digunakan!


In [5]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document

# --- FASE 2.5: LLM GRAPH BUILDER & ETL PIPELINE (TIER 4) ---
print("Memulai LLM Graph Builder...")

# 1. Extract: Teks laporan mentah
unstructured_text = """
Data intelijen ancaman siber terbaru:
Terdeteksi sebuah URL yaitu http://layanan-bca-prioritas.xyz/login
URL tersebut terhubung ke Domain layanan-bca-prioritas.xyz
Tim menetapkan klasifikasi URL tersebut dengan Label phishing
"""
documents = [Document(page_content=unstructured_text)]

llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["URL", "Domain", "Label"],
    allowed_relationships=["HAS_DOMAIN", "IS_CLASSIFIED_AS"]
)

print("AI sedang membaca teks dan mengekstrak entitas...")
graph_documents = llm_transformer.convert_to_graph_documents(documents)

# 2. Transform: Menormalkan format (Data Cleansing)
# Ini mengatasi isu bawaan LangChain yang selalu mengubah teks menjadi Title Case
for doc in graph_documents:
    # Perbaiki tipe label dan jadikan ID huruf kecil semua pada Node
    for node in doc.nodes:
        if node.type == "Url": 
            node.type = "URL"
        node.id = str(node.id).lower()
        
    # Lakukan hal yang sama pada garis Relasi
    for rel in doc.relationships:
        if rel.source.type == "Url": rel.source.type = "URL"
        rel.source.id = str(rel.source.id).lower()
        
        if rel.target.type == "Url": rel.target.type = "URL"
        rel.target.id = str(rel.target.id).lower()

# 3. Load: Masukkan data yang sudah divalidasi ke graf
graph.add_graph_documents(graph_documents)

print("✅ Ekstraksi dan Normalisasi selesai!")
print("Graf berhasil di-populate ke database dengan format yang konsisten.")

Memulai LLM Graph Builder...
AI sedang membaca teks dan mengekstrak entitas...


C:\Users\Shahnaz Ariqah\AppData\Local\Temp\ipykernel_18668\3781664568.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


✅ Ekstraksi dan Normalisasi selesai!
Graf berhasil di-populate ke database dengan format yang konsisten.


In [4]:
from langchain_neo4j import GraphCypherQAChain
from langchain_core.prompts import PromptTemplate
from IPython.display import display, Markdown # Tambahan wajib agar tampilan rapi di Jupyter

print("Membangun Agen AI Cybersecurity yang Sesungguhnya...")

# 1. Prompt Cypher (Pintar, General, Tanpa Cheat, tapi Sangat Ketat soal Sintaks!)
CYPHER_GENERATION_TEMPLATE = """Task: Generate a Cypher statement to query a graph database.
Schema:
{schema}

You are an expert Cypher developer. Translate the user's natural language question into ONE valid Cypher query based on the schema above.

General Logic for this Database:
1. To check the safety or classification of any specific link/domain: Traverse to the Label node. Pattern: (n)-[:IS_CLASSIFIED_AS]->(l:Label)
2. To find the "most", "top", or to count things (e.g., most used TLD): Use aggregation. Pattern: COUNT(x) ... ORDER BY ... DESC LIMIT 1
3. For unstructured entities (like news or company names): Use flexible relationships. Pattern: (n)-[r]-(m)
4. ALWAYS use `toLower()` and `CONTAINS` for string matching to avoid case sensitivity.

CRITICAL SYNTAX RULES (MUST OBEY):
- Generate EXACTLY ONE single Cypher query.
- NEVER generate multiple MATCH statements that have their own RETURNs. There must be ONLY ONE RETURN at the end of the query.
- DO NOT wrap the query in markdown blocks (do not use ```cypher).
- DO NOT output any explanation. Just the raw query.

Question: {question}
Cypher Query:"""

cypher_prompt = PromptTemplate(
    template=CYPHER_GENERATION_TEMPLATE,
    input_variables=["schema", "question"]
)

# 2. Prompt untuk Otak Analis (Mandiri & Fallback Cerdas)
QA_TEMPLATE = """You are a smart and professional Cybersecurity Analyst Copilot.
Use the following context retrieved from our Graph Database to answer the user's question.

Graph Database Context: 
{context}

RULES FOR ANSWERING:
1. If the context contains data, use it to give a definitive, factual answer.
2. IF THE CONTEXT IS EMPTY ([]), DO NOT say "I don't know". Instead:
   - State clearly that the URL/Domain/Entity is "not found in our historical database".
   - THEN, perform a real-time structural analysis of the user's input based on your general AI knowledge (e.g., explain the risks of the specific TLD, check for typosquatting, or explain common phishing tactics related to the question).
   - Give a final safety recommendation.
3. Format your answer beautifully using Markdown (bold text, bullet points).

User Question: {question}
Answer in Indonesian:"""

qa_prompt = PromptTemplate(
    template=QA_TEMPLATE,
    input_variables=["context", "question"]
)

# 3. Rakit Agen AI
chain = GraphCypherQAChain.from_llm(
    llm=llm, # Pastikan ini sudah pakai Groq Llama 3.3
    graph=graph,
    verbose=True, 
    cypher_prompt=cypher_prompt,
    qa_prompt=qa_prompt,
    allow_dangerous_requests=True,
    validate_cypher=True 
)

print("✅ Agen AI Cerdas Siap! Silakan coba link random apa pun.")
print("=" * 50)

while True:
    user_input = input("\nKamu: ")
    if user_input.lower() in ['keluar', 'exit', 'quit']:
        print("AI: Sesi diakhiri. Tetap waspada!")
        break
    if user_input.strip() == "":
        continue
        
    try:
        print("\n⏳ AI sedang menganalisis database dan ancaman...")
        jawaban = chain.invoke({"query": user_input})
        
        # --- TAMPILAN UI RAPI UNTUK JUPYTER NOTEBOOK ---
        print("\n" + "━" * 80)
        print("🛡️ CYBERSECURITY COPILOT RESPONSE")
        print("━" * 80)
        
        # Me-render output menjadi Markdown yang cantik
        display(Markdown(jawaban['result']))
        
        print("━" * 80 + "\n")
        
    except Exception as e:
        # Pesan error di-update sesuai Groq
        if "429" in str(e) or "rate_limit" in str(e).lower():
            print("\n🤖 AI: Limit API Groq tercapai. Tunggu sebentar ya.")
        else:
            print(f"\n🤖 AI: Error teknis: {e}")

Membangun Agen AI Cybersecurity yang Sesungguhnya...
✅ Agen AI Cerdas Siap! Silakan coba link random apa pun.

⏳ AI sedang menganalisis database dan ancaman...


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (u:URL)-[:IS_CLASSIFIED_AS]->(l:Label), (u)-[:HAS_TLD]->(t:TLD) WHERE toLower(l.id) CONTAINS 'phishing' RETURN t.id AS TLD, COUNT(u) AS count ORDER BY count DESC LIMIT 1
Full Context:
[{'TLD': '.com', 'count': 379}]

> Finished chain.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🛡️ CYBERSECURITY COPILOT RESPONSE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


**Analisis Top Level Domain (TLD) Phishing**
=============================================

Berdasarkan data yang tersedia di database kita, **.com** adalah Top Level Domain (TLD) yang paling banyak digunakan untuk phishing, dengan jumlah **379** kasus.

**Rincian Data:**
* TLD: **.com**
* Jumlah kasus: **379**

**Kesimpulan:**
TLD **.com** merupakan salah satu TLD yang paling umum digunakan, sehingga tidak mengherankan jika juga banyak digunakan untuk kegiatan phishing. Oleh karena itu, penting untuk tetap waspada dan melakukan verifikasi yang teliti saat menerima email atau mengunjungi situs web dengan TLD **.com**.

**Rekomendasi Keamanan:**
* Selalu verifikasi alamat email dan situs web sebelum mengklik tautan atau mengunduh file.
* Gunakan antivirus dan perangkat lunak keamanan yang terbaru untuk melindungi perangkat Anda.
* Jangan memberikan informasi pribadi atau keuangan tanpa memastikan bahwa situs web atau email tersebut aman dan terpercaya.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


⏳ AI sedang menganalisis database dan ancaman...


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (u:URL {id: toLower("https://broadwaystars.com/classic/?source=www.playbill.com")})-[:IS_CLASSIFIED_AS]->(l:Label) RETURN l.id
Full Context:
[]

> Finished chain.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🛡️ CYBERSECURITY COPILOT RESPONSE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


**Analisis URL broadwaystars.com/classic/?source=www.playbill.com**

Sayangnya, URL tersebut **tidak ditemukan di database historis** kami. Namun, kami dapat melakukan analisis struktural terhadap URL tersebut berdasarkan pengetahuan umum kami.

**Analisis Struktural:**

* Domain `broadwaystars.com` memiliki ekstensi TLD (Top-Level Domain) `.com`, yang umum digunakan untuk situs web bisnis dan komersial.
* Path `/classic/` menunjukkan bahwa URL tersebut mengarah ke halaman atau kategori "klasik" di dalam situs web.
* Parameter `?source=www.playbill.com` menunjukkan bahwa URL tersebut memiliki referensi dari situs web `playbill.com`, yang mungkin merupakan situs web teater atau hiburan.

**Risiko Potensial:**

* Typosquatting: Perlu diwaspadai kemungkinan adanya kesalahan penulisan domain atau path yang dapat mengarah ke situs web palsu atau berbahaya.
* Phishing: Parameter `?source=www.playbill.com` dapat digunakan untuk membuat URL tersebut terlihat lebih kredibel, namun perlu diwaspadai kemungkinan adanya serangan phishing yang menggunakan teknik ini.

**Rekomendasi Keamanan:**

* Pastikan untuk memeriksa ejaan domain dan path dengan teliti sebelum mengakses URL tersebut.
* Jangan mengklik pada link yang tidak dikenal atau tidak dipercaya, terutama jika link tersebut memiliki parameter yang mencurigakan.
* Gunakan perangkat lunak keamanan yang dapat mendeteksi dan mencegah serangan phishing dan malware.

Dengan demikian, kami sarankan untuk berhati-hati saat mengakses URL tersebut dan selalu memeriksa keamanan situs web sebelum melakukan interaksi apa pun.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

AI: Sesi diakhiri. Tetap waspada!
